# Lab 5: Feature Analysis

In this lab, you will transform numeric and categorical features and use Principal Component Analysis (PCA) to reduce a high-dimensional synthetic gene-expression dataset to two features.

## Learning objectives

- Apply min-max scaling and z-score standardization.
- Discretize numeric variables using multiple approaches.
- Transform skewed numeric data.
- Encode ordinal and nominal categorical variables.
- Standardize a high-dimensional dataset and reduce it to two principal components.
- Visualize and interpret PCA results.
- Optionally generate and select features.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 50)

## 2. Load and investigate the patient dataset

In [2]:
patient_data = pd.read_csv("data/patient_feature_transformations.csv")
patient_data.head()

,patient_id,age_years,biological_sex,height_cm,weight_kg,bmi,systolic_bp,diastolic_bp,fasting_glucose_mg_dl,crp_mg_l,cholesterol_mmol_l,activity_level,smoking_status,clinic_region,risk_group
0,P001,34,Male,178.4,94.99,29.83,126.2,68.3,81.5,2.188,4.67,Low,Former,West,Moderate
1,P002,82,Male,176.1,78.02,25.15,148.8,84.7,123.1,1.533,4.91,Low,Current,Central,Higher
2,P003,24,Female,155.4,65.23,27.02,93.2,66.8,85.8,0.776,3.53,Moderate,Never,East,Lower
3,P004,26,Male,176.5,84.16,27.02,132.6,73.1,83.0,1.000,4.22,Low,Current,East,Moderate
4,P005,57,Female,163.0,81.60,30.72,137.2,71.0,105.5,1.145,5.59,Low,Never,West,Higher


In [3]:
patient_data.shape
patient_data.dtypes
patient_data.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
patient_id,180,180,P001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age_years,180.0,NaN,NaN,NaN,52.511111,19.352504,18.0,37.0,53.0,69.25,85.0
biological_sex,180,2,Female,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
height_cm,180.0,NaN,NaN,NaN,170.502778,9.886221,148.9,163.1,170.0,177.5,197.7
weight_kg,180.0,NaN,NaN,NaN,78.729667,17.273135,41.02,65.905,77.635,89.78,125.18
bmi,180.0,NaN,NaN,NaN,26.963056,4.865772,17.0,23.525,26.985,30.74,38.59
systolic_bp,180.0,NaN,NaN,NaN,128.279444,13.870328,93.2,118.45,127.75,137.875,162.3
diastolic_bp,180.0,NaN,NaN,NaN,72.541111,6.856003,54.4,67.275,73.15,77.5,89.1
fasting_glucose_mg_dl,180.0,NaN,NaN,NaN,92.016111,12.523257,64.2,82.675,91.1,100.0,127.8
crp_mg_l,180.0,NaN,NaN,NaN,2.049117,1.888977,0.194,0.90375,1.5135,2.4495,13.268


### Dataset investigation

Investigate the dataset and describe what the features are measuring. Identify which columns are:

- identifiers;
- numeric features;
- ordinal categorical features;
- nominal categorical features;
- the target column.

**Your notes:**



Identifier:
- patient_id

Numeric:
- age_years
- height_cm
- weight_kg
- bmi
- systolic_bp
- diastolic_bp
- fasting_glucose_mg_dl
- crp_mg_l
- cholesterol_mmol_l

Ordinal Categorical:
- activity_level
- smoking_status

Nominal Categorical:
- clinic_region

Target Column:
- risk_group

## 3. Numeric feature transformations

### 3.1 Rounding

Create a rounded version of `weight_kg`. Compare the original and rounded values. Discuss one benefit and one drawback of rounding.

Suggested functions: `.round()` or `np.round()`

In [5]:
# Created a rounded version of the weight_kg column.
patient_data["weight_kg_rounded"] = patient_data["weight_kg"].round(0)
patient_data[["weight_kg", "weight_kg_rounded"]].head()

# A drawba

,weight_kg,weight_kg_rounded
0,94.99,95.0
1,78.02,78.0
2,65.23,65.0
3,84.16,84.0
4,81.60,82.0


A benefit of rounding is that the data is more readable and easier to interpret. Furthermore, rounding may be useful if one wishes to assign weight categories based on integer ranges or if further processing requires an integer value.

A drawback of rounding is that some information is invariably lost.

### 3.2 Min-max scaling

Use `MinMaxScaler` to transform the following columns to the range 0 to 1:

- `age_years`
- `bmi`
- `systolic_bp`
- `fasting_glucose_mg_dl`

Store the transformed values in a new DataFrame with clear column names. Display summary statistics for the transformed columns.

In [10]:
minmax_columns = [
    "age_years",
    "bmi",
    "systolic_bp",
    "fasting_glucose_mg_dl",
]

# For the listed columns, apply the MinMaxScaler to scale the values between 0 and 1.
# Store the transformed values in a new DataFrame.
minmax_scaler = MinMaxScaler()
patient_data_minmax_scaled = pd.DataFrame(
    minmax_scaler.fit_transform(patient_data[minmax_columns]),
    columns=[col + "_minmax" for col in minmax_columns],
)

# Display summary statistics for the MinMax scaled columns.
patient_data_minmax_scaled.describe().T

,count,mean,std,min,25%,50%,75%,max
age_years_minmax,180.0,0.515091,0.288843,0.0,0.283582,0.522388,0.764925,1.0
bmi_minmax,180.0,0.461466,0.225372,0.0,0.302223,0.462483,0.636406,1.0
systolic_bp_minmax,180.0,0.507662,0.200728,0.0,0.365412,0.500000,0.646527,1.0
fasting_glucose_mg_dl_minmax,180.0,0.437360,0.196907,0.0,0.290487,0.422956,0.562893,1.0


### 3.3 Z-score standardization

Use `StandardScaler` on the same numeric columns. Confirm that the transformed columns have means close to 0 and standard deviations close to 1.

In [13]:
# Use StandardScaler on the numeric columns.
standard_scaler = StandardScaler()
patient_data_standard_scaled = pd.DataFrame(
    standard_scaler.fit_transform(patient_data[minmax_columns]),
    columns=[col + "_standard" for col in minmax_columns],
)
patient_data_standard_scaled.describe().T

,count,mean,std,min,25%,50%,75%,max
age_years_standard,180.0,7.894919e-17,1.002789,-1.788263,-0.803740,0.025333,0.867360,1.683478
bmi_standard,180.0,-9.868649e-18,1.002789,-2.053291,-0.708551,0.004523,0.778392,2.396203
systolic_bp_standard,180.0,-3.873445e-16,1.002789,-2.536155,-0.710644,-0.038277,0.693734,2.459600
fasting_glucose_mg_dl_standard,180.0,2.072416e-16,1.002789,-2.227352,-0.747982,-0.073357,0.639303,2.865365


### 3.4 Compare the transformations

Choose one numeric feature and plot its:

1. original values;
2. min-max scaled values;
3. standardized values.

Use histograms and explain what changed and what stayed the same.

### 3.5 Equal-width discretization

Divide `age_years` into equal-width bins. Add the result as a new column and count the number of patients in each bin.

### 3.6 Equal-frequency discretization

Divide `age_years` into groups containing approximately equal numbers of patients. Compare the group counts with the equal-width result.

### 3.7 Meaningful threshold-based discretization

Create three classroom categories from `fasting_glucose_mg_dl` using thresholds that you choose. Give the categories meaningful names and explain your choices.

### 3.8 Log transformation

Find a column which should be log transformed (skewed distribution) and create a new column with its log tranformation then compare the two columns.


## 4. Categorical feature transformations

### 4.1 Ordinal encoding

Encode `activity_level` into a numeric feature

### 4.2 One-hot encoding

Encode `smoking_status` and `clinic_region` into numeric features.

### 4.3 Reflection

What encoding style did you choose for each column and why?

**Your answer:**



## 5. Principal Component Analysis

The second dataset contains 40 synthetic gene-expression features. PCA will be used to replace these 40 features with two principal components.

The `diagnosis_group` column will be used only to colour the final plot. PCA itself does not use the labels.

In [ ]:
gene_data = pd.read_csv("data/gene_expression_pca.csv")
gene_data.head()

In [ ]:
gene_data.shape
gene_data["diagnosis_group"].value_counts()


### 5.1 Separate the features and labels

Create:

- `X_gene`: only the gene-expression columns;
- `y_gene`: the `diagnosis_group` column.

Do not include `sample_id` in the PCA input.

### 5.2 Standardize the gene-expression features

Use `StandardScaler` to standardize all gene-expression columns. Store the result in `X_gene_scaled`.

### 5.3 Reduce the dataset to two principal components

Create `PCA(n_components=2)`, fit it to the standardized gene-expression data, and transform the data. Store the two components in a DataFrame named `pca_data` with columns `PC1` and `PC2`.

Add `diagnosis_group` to the PCA DataFrame after the transformation.

### 5.4 Explained variance

Display `explained_variance_ratio_` and calculate the total proportion of variance explained by the first two components.

### 5.5 Visualize the two-component representation

Create a scatter plot with:

- PC1 on the x-axis;
- PC2 on the y-axis;
- a different colour and legend entry for each diagnosis group.

The plot should show whether the original 40-feature dataset can be represented in two dimensions while preserving visible group separation.

### 5.6 PCA interpretation

Answer the following:

1. Do the three diagnosis groups appear separated?
2. How much variance is explained by PC1 and PC2 together?
3. Are PC1 and PC2 original genes or newly created features?
4. Why was the data standardized before PCA?

**Your answers:**



## 6. Bonus: Feature generation

Create at least two new features from the patient dataset. Possible ideas include:

- pulse pressure: systolic blood pressure minus diastolic blood pressure;
- an age group;
- a high-measurement indicator;
- a ratio or interaction between two measurements.

Explain what each new feature represents and why it might be useful.

## 7. Bonus: Feature selection

Try one or both of the following:

1. Use `VarianceThreshold` to identify the deliberately low-variance gene features.
2. Use `SelectKBest` with `f_classif` to select genes that are strongly associated with `diagnosis_group`.

In [ ]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

# Complete one or both feature-selection activities below.


## 8. Final reflection

Describe one situation where each method could be useful:

- min-max scaling;
- standardization;
- discretization;
- one-hot encoding;
- PCA.

Also describe one piece of information that can be lost during transformation or dimensionality reduction.

**Your response:**

